<a href="https://colab.research.google.com/github/EbrahemOsama22/AirLine_Delay/blob/main/BERT_Sentiment_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##install libraries


In [ ]:
!pip install transformers datasets torch scikit-learn optimum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 1.1 MB/s eta 0:00:00


In [1]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/sst2")

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

#الـ AutoTokenizer هو الأداة اللي بتحول النص لأرقام
#والـ AutoModelForSequenceClassification هو الموديل (العقل) اللي بيفهم النص ويصنفه.
#الموديل المختار: bert-base-multilingual-cased
#ودا نموذج بيرت  مدربه ومعمول ليها توكنيزر

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-uncased", num_labels=2)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


##AutoTokenizer->دي بس بتحمّل الأداة (الـ Tokenizer)
##tokenizer->دي بتاخد الداتا كلها وبتحوّلها



In [13]:
def preprocess_function(examples):
    return tokenizer(examples["sentence"],
                     padding="max_length",
                     truncation=True,
                     max_length=128)

tokenized_datasets = ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [16]:
print(ds)
print(ds['train'][0])


DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})
{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}


In [42]:
# الخطوة 4 - Train/Test Split 80/20
split = tokenized_datasets["train"].train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
test_dataset = split["test"]
val_dataset = tokenized_datasets["validation"]

In [48]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=10,
    fp16=True,
)


In [50]:
import numpy as np
from sklearn.metrics import accuracy_score

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [51]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.129086,0.268751,0.930046
2,0.074184,0.293441,0.927752
3,0.042679,0.366221,0.927752


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=10104, training_loss=0.0862517739960541, metrics={'train_runtime': 3383.982, 'train_samples_per_second': 47.765, 'train_steps_per_second': 2.986, 'total_flos': 1.063212041380608e+16, 'train_loss': 0.0862517739960541, 'epoch': 3.0})

In [52]:
# الخطوة 9 - التقييم على Test
results = trainer.evaluate(test_dataset)
print(f"Test Accuracy: {results['eval_accuracy']:.4f}")


Training Loss,Validation Loss,Epoch,Accuracy
0.042679,0.101120,3,0.971269


Test Accuracy: 0.9713


In [53]:
# الخطوة 10 - حفظ الموديل
trainer.save_model("./finetuned_bert_sst2")
tokenizer.save_pretrained("./finetuned_bert_sst2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./finetuned_bert_sst2/tokenizer_config.json',
 './finetuned_bert_sst2/tokenizer.json')

In [54]:
texts = [
    "This movie is great!",
    "This movie is terrible!",
    "The acting was wonderful",
    "I hated this film",
    "It was okay, not great not bad",
]



In [57]:
from transformers import pipeline

classifier = pipeline("text-classification", model="./finetuned_bert_sst2", tokenizer="./finetuned_bert_sst2")

for text in texts:
    result = classifier(text)
    label = "Positive" if result[0]["label"] == "LABEL_1" else "Negative"
    score = result[0]["score"]
    print(f"Text: {text}")
    print(f"Prediction: {label} ({score:.4f})")
    print("-" * 50)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: This movie is great!
Prediction: Positive (0.9972)
--------------------------------------------------
Text: This movie is terrible!
Prediction: Negative (0.9971)
--------------------------------------------------
Text: The acting was wonderful
Prediction: Positive (0.9975)
--------------------------------------------------
Text: I hated this film
Prediction: Negative (0.9970)
--------------------------------------------------
Text: It was okay, not great not bad
Prediction: Positive (0.9969)
--------------------------------------------------
